In [17]:
# %% Setup
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df = pd.read_csv("../data/raw/dirty_cafe_sales.csv")   # path is relative to notebooks/

In [18]:
# %% 0. Rows and columns
print(df.shape)
df.head()

(10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [19]:
# %% 1. Dtypes: which ones look wrong?
expected = {
    "Transaction ID": "str",
    "Item": "str",
    "Quantity": "int64",
    "Price Per Unit": "float64",
    "Total Spent": "float64",
    "Payment Method": "str",
    "Location": "str",
    "Transaction Date": "datetime64[ns]",
}
pd.DataFrame({"actual": df.dtypes.astype(str), "expected": pd.Series(expected)}) \
  .assign(wrong=lambda t: t.actual != t.expected)

,actual,expected,wrong
Transaction ID,str,str,False
Item,str,str,False
Quantity,str,int64,True
Price Per Unit,str,float64,True
Total Spent,str,float64,True
Payment Method,str,str,False
Location,str,str,False
Transaction Date,str,datetime64[ns],True


In [20]:
# Simpler: the mean of booleans is already a proportion
df.isna().mean().mul(100).round(1)

# Wrap in parentheses instead of using a backslash (PEP 8)
(
    pd.DataFrame({"actual": df.dtypes.astype(str), "expected": pd.Series(expected)})
    .assign(wrong=lambda t: t.actual != t.expected)
)

,actual,expected,wrong
Transaction ID,str,str,False
Item,str,str,False
Quantity,str,int64,True
Price Per Unit,str,float64,True
Total Spent,str,float64,True
Payment Method,str,str,False
Location,str,str,False
Transaction Date,str,datetime64[ns],True


In [21]:
# %% 2. Nulls per column
df.isna().sum().to_frame("nulls").assign(pct=lambda t: (t.nulls / len(df) * 100).round(1))

,nulls,pct
Transaction ID,0,0.0
Item,333,3.3
Quantity,138,1.4
Price Per Unit,179,1.8
Total Spent,173,1.7
Payment Method,2579,25.8
Location,3265,32.6
Transaction Date,159,1.6


In [22]:
import numpy as np

# %% 3. Duplicates
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicate Transaction IDs:", df["Transaction ID"].duplicated().sum())

MISSING_TOKENS = {"error", "unknown", ""}

def standardize_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Replace junk placeholders ("ERROR", "UNKNOWN", "") with NaN."""
    df = df.copy()

    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        normalized = df[col].astype("string").str.strip().str.lower()
        df[col] = df[col].mask(normalized.isin(MISSING_TOKENS), np.nan)

    return df

print(standardize_missing(df))

Fully duplicated rows: 0
Duplicate Transaction IDs: 0
     Transaction ID      Item Quantity Price Per Unit Total Spent  Payment Method  Location Transaction Date
0       TXN_1961373    Coffee        2            2.0         4.0     Credit Card  Takeaway       2023-09-08
1       TXN_4977031      Cake        4            3.0        12.0            Cash  In-store       2023-05-16
2       TXN_4271903    Cookie        4            1.0         NaN     Credit Card  In-store       2023-07-19
3       TXN_7034554     Salad        2            5.0        10.0             NaN       NaN       2023-04-27
4       TXN_3160411    Coffee        2            2.0         4.0  Digital Wallet  In-store       2023-06-11
...             ...       ...      ...            ...         ...             ...       ...              ...
9995    TXN_7672686    Coffee        2            2.0         4.0             NaN       NaN       2023-08-30
9996    TXN_9659401       NaN        3            NaN         3.0  Digital

In [23]:
# %% 4. Unique values in the text columns
raw = pd.read_csv("../data/raw/dirty_cafe_sales.csv", dtype=str, keep_default_na=False)

for col in ["Item", "Payment Method", "Location"]:
    print(f"\n--- {col} ---")
    print(raw[col].value_counts())


--- Item ---
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
             333
ERROR        292
Name: count, dtype: int64

--- Payment Method ---
Payment Method
                  2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

--- Location ---
Location
            3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64


In [7]:
# %% 4b. Junk values hiding in every column (including the "numeric" ones)
junk = ["", "ERROR", "UNKNOWN"]
raw.apply(lambda s: s.isin(junk).sum()).to_frame("junk_count")

,junk_count
Transaction ID,0
Item,969
Quantity,479
Price Per Unit,533
Total Spent,502
Payment Method,3178
Location,3961
Transaction Date,460


In [24]:
cleaned = standardize_missing(df)

print((df == "ERROR").sum().sum())       # should be > 0: original untouched
print((cleaned == "ERROR").sum().sum())  # should be 0: junk is gone
print(cleaned.isna().mean().mul(100).round(1))  # Item should be ~9.7%

1622
0
Transaction ID       0.0
Item                 9.7
Quantity             4.8
Price Per Unit       5.3
Total Spent          5.0
Payment Method      31.8
Location            39.6
Transaction Date     4.6
dtype: float64
